In [24]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression

In [25]:
df = sns.load_dataset("tips")
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [26]:
df = df[["total_bill", "tip"]]
df.head(3)

,total_bill,tip
0,16.99,1.01
1,10.34,1.66
2,21.01,3.50


In [27]:
from sklearn.model_selection import train_test_split

In [28]:
# independed and dependent feature split
X = df.drop("tip", axis=1)
y = df["tip"]
type(X), type(y)

(pandas.core.frame.DataFrame, pandas.core.series.Series)

In [29]:
# train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=42
)
# check if X_train and y_traain has same number of values
# check if x_test and y_test has same number of values
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((195, 1), (49, 1), (195,), (49,))

In [30]:
lr_model = LinearRegression()
# fit the model
lr_model.fit(X_train, y_train)

LinearRegression()

In [31]:
predictions = lr_model.predict(X_test) # return predictions
predictions

array([3.04525623, 1.86330727, 3.55119456, 3.69452593, 2.31576375,
       2.83881627, 3.96728338, 2.26014262, 2.50615915, 2.57033737,
       2.88160176, 2.07723468, 2.06439904, 2.47407003, 2.00236009,
       2.91903905, 2.92652651, 3.23351235, 2.68478854, 5.33107064,
       3.13831465, 3.13403611, 2.4558862 , 1.94673896, 3.16077703,
       2.17564129, 2.02375283, 3.62927807, 2.68906708, 6.07767732,
       4.99734388, 1.75313465, 2.83025918, 3.09552917, 2.74040966,
       3.50092162, 2.21200895, 5.53644096, 2.33287794, 3.35010279,
       2.04942412, 2.47834858, 3.48701634, 2.03017065, 2.03124029,
       1.25361414, 2.05798121, 2.92438724, 1.73388118])

In [32]:
X_test.loc[24]

total_bill    19.82
Name: 24, dtype: float64

In [33]:
lr_model.predict([X_test.loc[24]])
#to get one value 

d:\anoconda\envs\tech-axis\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([3.04525623])

In [34]:
test = pd.DataFrame({
    'total_bill': X_test['total_bill'],
    'tip': y_test,
    'predictions': predictions
})
test.head()
# gives approximate values of tips based on total_bill

,total_bill,tip,predictions
24,19.82,3.18,3.045256
6,8.77,2.00,1.863307
153,24.55,2.00,3.551195
211,25.89,5.16,3.694526
198,13.00,2.00,2.315764


In [35]:
test['diff'] = test['predictions'] - test['tip']
test.head()

,total_bill,tip,predictions,diff
24,19.82,3.18,3.045256,-0.134744
6,8.77,2.00,1.863307,-0.136693
153,24.55,2.00,3.551195,1.551195
211,25.89,5.16,3.694526,-1.465474
198,13.00,2.00,2.315764,0.315764


In [36]:
from sklearn.metrics import root_mean_squared_error, r2_score
# mean squared error: average of the squared differences between predicted and actual values
root_mean_squared_error(y_test, predictions)

0.7541977545199626

In [37]:
r2_score(y_test, predictions)
# r2_score: shows how good the model is at predicting the values

0.5449381659234663

In [45]:
np.random.seed(42)

class LinearRegressionCustom:
    def __init__(self, learning_rate=0.001, epoch = 100, bias=True) -> None:
        self.learning_rate = learning_rate
        self.epoch = epoch
        self.bias = bias
        # self.w = None

    def train(self, X: pd.DataFrame, y: pd.Series):
        # first check if the input is a pandas DataFrame or a numpy array
        if isinstance(X, pd.DataFrame):
            _X = X.to_numpy()
        elif not isinstance(X, np.ndarray):
            raise ValueError("X must be a pandas DataFrame or a 2D numpy array")
        
        if isinstance(y, pd.Series):
            _y = y.to_numpy()
        elif not isinstance(y, np.ndarray):
            raise ValueError("y must be a pandas DataFrame or a 1D numpy array")
        
        # find num of records and num features
        n_recs, n_features = _X.shape

        # initialize the weights
        self.w = np.random.random(n_features)  

        # epoch: fit the entire data
        for epoch in range(1, self.epoch + 1 ):
            y_hat = self.predict(_X)

            diff = _y - y_hat

            loss = self.loss_mse(diff, n_recs)

            # calculate the gradient
            # dot product used because it is doing the job of summation 
            # gradient direction is away from the minimum point so minus sign is used
            grad_w = -(2 / n_recs) * np.dot(_X.T, diff) 
            # not using dot product, so we need to sum the diff values for bias
            grad_b = -(2 / n_recs) * np.sum(diff)

            self.w = self.w - self.learning_rate * grad_w
            self.bias = self.bias - self.learning_rate * grad_b

            # if epoch % 10 == 0:
            print(f"Epoch: {epoch}, Loss: {loss}")

    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()
        elif not isinstance(X, np.ndarray):
            raise ValueError("X must be a pandas DataFrame or a 2D numpy array")
        # dot prod = input feature to corresponding weights multiplication
        return np.dot(X, self.w) + self.bias
    
    def loss_mse(self, diff, n_recs):
        return (1 / n_recs) * np.sum(np.square(diff))


In [46]:
lr_model_custom = LinearRegressionCustom()
lr_model_custom.train(X_train, y_train)

Epoch: 1, Loss: 36.71547603195486
Epoch: 2, Loss: 1.18415906676482
Epoch: 3, Loss: 1.1567970935446248
Epoch: 4, Loss: 1.1567756233672137
Epoch: 5, Loss: 1.1567752075211126
Epoch: 6, Loss: 1.156774808139709
Epoch: 7, Loss: 1.15677440902223
Epoch: 8, Loss: 1.1567740101560424
Epoch: 9, Loss: 1.1567736115409781
Epoch: 10, Loss: 1.1567732131768795
Epoch: 11, Loss: 1.1567728150635879
Epoch: 12, Loss: 1.1567724172009457
Epoch: 13, Loss: 1.1567720195887952
Epoch: 14, Loss: 1.1567716222269786
Epoch: 15, Loss: 1.1567712251153381
Epoch: 16, Loss: 1.1567708282537166
Epoch: 17, Loss: 1.1567704316419563
Epoch: 18, Loss: 1.1567700352799
Epoch: 19, Loss: 1.1567696391673905
Epoch: 20, Loss: 1.1567692433042709
Epoch: 21, Loss: 1.1567688476903841
Epoch: 22, Loss: 1.1567684523255728
Epoch: 23, Loss: 1.1567680572096808
Epoch: 24, Loss: 1.156767662342551
Epoch: 25, Loss: 1.1567672677240266
Epoch: 26, Loss: 1.1567668733539516
Epoch: 27, Loss: 1.156766479232169
Epoch: 28, Loss: 1.1567660853585229
Epoch: 29, L

In [47]:
custom_preds = lr_model_custom.predict(X_test)
custom_preds

array([3.05596301, 1.9023513 , 3.54977147, 3.68966646, 2.34396013,
       2.85447246, 3.95588455, 2.28967252, 2.52979079, 2.59243034,
       2.89623216, 2.1111498 , 2.09862189, 2.49847102, 2.03807033,
       2.9327719 , 2.94007985, 3.23970569, 2.70413754, 5.28697499,
       3.14679036, 3.14261439, 2.48072315, 1.98378272, 3.1687142 ,
       2.20719711, 2.05895018, 3.62598292, 2.70831351, 6.01568175,
       4.96124933, 1.79482007, 2.84612052, 3.10503066, 2.75842515,
       3.50070382, 2.24269286, 5.48742155, 2.36066401, 3.35350088,
       2.084006  , 2.50264699, 3.48713192, 2.06521413, 2.06625812,
       1.30727558, 2.09235794, 2.93799186, 1.77602821])

In [48]:
pd.DataFrame({
    'total_bill': X_test['total_bill'],
    'tip': y_test,
    'predictions': predictions,
    'custom_preds': custom_preds
})

,total_bill,tip,predictions,custom_preds
24,19.82,3.18,3.045256,3.055963
6,8.77,2.00,1.863307,1.902351
153,24.55,2.00,3.551195,3.549771
211,25.89,5.16,3.694526,3.689666
198,13.00,2.00,2.315764,2.343960
176,17.89,2.00,2.838816,2.854472
192,28.44,2.56,3.967283,3.955885
124,12.48,2.52,2.260143,2.289673
9,14.78,3.23,2.506159,2.529791
101,15.38,3.00,2.570337,2.592430


In [49]:
r2_score(y_test, custom_preds)

0.5437076497613581